# Data Collection

In [ ]:
import pandas as pd
import numpy as np

##########################################
'''DATA'''
##########################################
alpha = 0.3 #TODO Trouver une source qui justifie ce choix 
Tmax= 15 #au dessus : pas de chauffage
Tmin = 0 #en dessous: chauffage maximal
cos_phi = 0.95
delta_Umin = 0.95 #TODO changer avec la bonne valeur selon le type de cable
delta_Umax = 1.05 #TODO changer avec la bonne valeur selon le type de cable
V = 20 #kV

##########################################
# 1. Charger data du fichier 87_grid
file = "87_0_grid.xlsx"

parameters = pd.read_excel(file, sheet_name="parameters")
f =  parameters["f_hz"].values[0] 
omega = 2 * np.pi * f

base = pd.read_excel(file, sheet_name="res_ext_grid") #TODO : CHANGER AVEC NOUVEAU OPF POUR SCENARIO 2 
P_base = base["p_mw"].values
Q_base = base["q_mvar"].values

load = pd.read_excel(file, sheet_name="load")
P_load_agg = load.groupby("bus")["p_mw"].sum() #tableau avec pour chaque node une valeur de load associée 
Q_load_agg = load.groupby("bus")["q_mvar"].sum()

bus = pd.read_excel(file, sheet_name="bus")
n_nodes = len(bus)
nodes = range(n_nodes)
P_load = {k: P_load_agg.get(k, 0) for k in nodes}
Q_load = {k: Q_load_agg.get(k, 0) for k in nodes}

bus_ref = pd.read_excel(file, sheet_name="res_bus") #TODO : CHANGER AVEC NOUVEAU OPF POUR SCENARIO 2 
P_ref = bus_ref["p_mw"].values
Q_ref = bus_ref["q_mvar"].values

trafo = pd.read_excel(file, sheet_name="trafo")
pcc_bus = int(trafo["lv_bus"].iloc[0])

lines = pd.read_excel(file, sheet_name="line")
#lines["busi - busj"] = lines["from_bus"].astype(str) + "-" + lines["to_bus"].astype(str)
line_data = pd.DataFrame({
    #"id": lines["id"],
    "from_bus": lines["from_bus"],
    "to_bus": lines["to_bus"],
    "length": lines["length_km"],
    "r": lines["r_ohm_per_km"] * lines["length_km"],
    "x": lines["x_ohm_per_km"] * lines["length_km"],
    "c": lines["c_nf_per_km"] * lines["length_km"] * 1e-9,
    "I_max": lines["max_i_ka"]
})

#Construire gij, bij, bij(sh)
line_data["g"] = line_data["r"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b"] = -line_data["x"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b_sh"] = omega * line_data["c"]
#print(line_data.head()) 

#Construire les matrices Jacobiennes
# Initialisation des matrices
J_Ptheta = np.zeros((n_nodes, n_nodes))
J_QU     = np.zeros((n_nodes, n_nodes))
J_PU     = np.zeros((n_nodes, n_nodes))
# Boucle sur chaque ligne ij
for l in range(len(line_data)):
    i = int(line_data.loc[l, "from_bus"])
    j = int(line_data.loc[l, "to_bus"])

    gij = line_data.loc[l, "g"]
    bij = line_data.loc[l, "b"]
    bsh = line_data.loc[l, "b_sh"]
# 1. MATRICE J_Ptheta
    # diagonale
    J_Ptheta[i, i] -= bij
    J_Ptheta[j, j] -= bij
    # hors diagonale
    J_Ptheta[i, j] += bij
    J_Ptheta[j, i] += bij
# 2. MATRICE J_QU
    # diagonale
    J_QU[i, i] -= (2*bsh + bij)
    J_QU[j, j] -= (2*bsh + bij)
    # hors diagonale
    J_QU[i, j] += bij
    J_QU[j, i] += bij
# 3. MATRICE J_PU
    # diagonale
    J_PU[i, i] += gij
    J_PU[j, j] += gij
    # hors diagonale
    J_PU[i, j] -= gij
    J_PU[j, i] -= gij
# 4. MATRICE J_Qtheta
J_Qtheta = - J_PU
#Calculer S_max 
line_data["S_max"] = np.sqrt(3) * line_data["I_max"] * V

##########################################
# 2. Charger data des PV
irr = pd.read_csv("irradiance_hourly.csv") #TODO changer avec le bon fichier 
G = irr["irradiance_W_m2"].values  # taille 8760
G_norm = G / np.max(G)

pv_data = []    #Data PV par load 
pv_data_dt = [] #Data PV par load par heure de l'année 
for k in range(n_nodes):
    Pk = P_load_agg.get(k, 0) # load au node k
    Ppv = alpha * Pk #Ppv au node k 
    Qpv = Ppv * np.tan(np.arccos(cos_phi))
    pv_data.append({
        "bus": k,
        "P_pv": Ppv,
        "Q_pv": Qpv
    })
    for t in range(len(G)):
        Ppv = alpha * Pk * G_norm[t]
        Qpv = Ppv * np.tan(np.arccos(cos_phi))
        pv_data_dt.append({
            "bus": k,
            "time": t,
            "P_pv": Ppv,
            "Q_pv": Qpv
        })
pv_df = pd.DataFrame(pv_data)
P_pv_max = pv_df.set_index("bus")["P_pv"]
Q_pv_max = pv_df.set_index("bus")["Q_pv"]
pv_df.to_csv("pv_data.csv", index=False)
pv_df_dt = pd.DataFrame(pv_data_dt)
pv_df_dt.to_csv("pv_data_dt.csv", index=False)

##########################################
# 3. Charger data des HP
COP = 3.9         # heat pump (nPro) #TODO changer quand zone bien délimitée
P_hp_total = 26072 / 1000 # MW thermique (nPro) #TODO changer quand zone bien délimitée
temp = pd.read_csv("temperature_hourly.csv") #TODO changer avec bon fichier

T = temp["temperature_C"].values
f_t = np.clip((Tmax - T) / (Tmax - Tmin), 0, 1) # Fonction chauffage qui indique consommation de l'HP en fonction de la temperature exterieure 

P_hp_total_elec = P_hp_total / COP #puissance electrique totale 
Pk_total = P_load_agg.sum() # Total load

hp_data = []
hp_data_dt = []
for k in range(n_nodes):
    Pk = P_load_agg.get(k, 0)
    P_hp = P_hp_total_elec * (Pk / Pk_total)
    hp_data.append({
        "bus": k,
        "P_hp": P_hp
    })
    for t in range(len(T)):
        P_hp_dt = P_hp * f_t[t]
        hp_data_dt.append({
            "bus": k,
            "time": t,
            "P_hp": P_hp_dt
        })
hp_df = pd.DataFrame(hp_data)
P_hp_max = hp_df.set_index("bus")["P_hp"] #Structure les données pour Gurobi
hp_df.to_csv("hp_data.csv", index=False)
hp_df_dt = pd.DataFrame(hp_data_dt)
hp_df_dt.to_csv("hp_data_dt.csv", index=False)


PCC bus = 19


# CREE LE MODELE FFOR

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# =========================
# IMPORT DATA
# =========================
# data = dp.get_data()
# P_base   = data["P_base"]
# Q_base   = data["Q_base"]
# P_load   = data["P_load"]
# Q_load   = data["Q_load"]
# nodes    = data["nodes"]
# n_nodes  = data["n_nodes"]
# P_ref    = data["P_ref"]
# Q_ref    = data["Q_ref"]
# J_Ptheta = data["J_Ptheta"]
# J_PU     = data["J_PU"]
# J_QU     = data["J_QU"]
# line_data = data["line_data"]
# pv_df    = data["pv_df"]
# pv_df_dt = data["pv_df_dt"]
# hp_df    = data["hp_df"]
# hp_df_dt = data["hp_df_dt"]

# =========================
# MODELE
# =========================

model = gp.Model("FFOR_static")

# =========================
# VARIABLES
# =========================
Pk = model.addVars(n_nodes, lb=-GRB.INFINITY, name="Pk")
Qk = model.addVars(n_nodes, lb=-GRB.INFINITY, name="Qk")
theta = model.addVars(n_nodes, lb=-GRB.INFINITY, name="theta")
delta_U     = model.addVars(n_nodes, lb=-GRB.INFINITY, name="U")
P_pv = model.addVars(n_nodes, lb=0, name="Ppv")
Q_pv = model.addVars(n_nodes, lb=-GRB.INFINITY, name="Qpv")
P_hp = model.addVars(n_nodes, lb=-GRB.INFINITY, name="Qpv")

# =========================
# CONTRAINTES FFOR
# =========================

# Nodal Power Balance
for k in nodes:
    model.addConstr(
        Pk[k] == P_pv.get(k, 0) - P_hp.get(k, 0) - P_load.get(k, 0),
        name=f"Nodal_P_balance_{k}"
    )
    model.addConstr(
        Qk[k] == Q_pv.get(k, 0) - Q_load.get(k, 0),
        name=f"Nodal_Q_balance_{k}"
    )

#Linearized Power Flow
for k in nodes:
    model.addConstr(
        Pk[k] == P_ref[k]
        +
        gp.quicksum(J_Ptheta[k, j] * theta[j] for j in nodes)
        +
        gp.quicksum(J_PU[k, j] * delta_U[j] for j in nodes),
        name=f"Power_flow_P{k}"
    )
    model.addConstr(
        Qk[k] == Q_ref[k]
        +
        gp.quicksum(J_Qtheta[k, j] * theta[j] for j in nodes)
        +
        gp.quicksum(J_QU[k, j] * delta_U[j] for j in nodes),
        name=f"Power_flow_Q{k}"
    )

#PV constraint 
for k in nodes:
    model.addConstr(P_pv[k] <= P_pv_max.get(k, 0),
        name=f"PV_Pmax_pos_{k}"
    )
    model.addConstr(0 <= P_pv[k],
        name=f"PV_Pmax_neg_{k}"
    )
    model.addConstr(Q_pv[k] <= Q_pv_max[k],
        name=f"PV_Qmax_pos_{k}"
    )
    model.addConstr(Q_pv[k] >= -Q_pv_max[k],
        name=f"PV_Qmax_neg_{k}"
    )

#HP constraint 
for k in nodes:
    model.addConstr(P_hp[k] <= P_hp_max.get(k, 0),
        name=f"HP_Pmax_pos_{k}"
    )
    model.addConstr(0 <= P_hp[k],
        name=f"HP_Pmax_neg_{k}"
    )

#Line flow constraint
for idx, row in line_data.iterrows():
    i = int(row["from_bus"])
    j = int(row["to_bus"])
    b = row["b"]
    Smax = row["S_max"]
    Pij = b * (theta[i] - theta[j]) #TODO verifier cette equation
    Qij = b * (delta_U[i] - delta_U[j]) #TODO verifier cette equation

    model.addQConstr(
        Pij * Pij + Qij * Qij <= Smax * Smax,
        name=f"line_flow_{i}_{j}"
    )

#Voltage Constraint
for k in nodes:
    model.addConstr(
        V + delta_U[k] >= delta_Umin*V,
        name=f"Umin_{k}"
    )
    model.addConstr(
        V + delta_U[k] <= delta_Umax*V,
        name=f"Umax_{k}"
    )

#Slack Constraint
# =========================
# SLACK BUS (PCC)
# =========================
model.addConstr(
    theta[pcc_bus] == 0,
    name="Slack_theta"
)

# Tension fixée (ΔU = 0)
model.addConstr(
    delta_U[pcc_bus] == 0,
    name="Slack_voltage"
)
# =========================
# SLACK NODE
# =========================

slack = 0
model.addConstr(theta[slack] == 0)

# =========================
# OBJECTIF
# =========================

model.setObjective(
    gp.quicksum(theta[k]*theta[k] for k in nodes) +
    gp.quicksum(U[k]*U[k] for k in nodes),
    GRB.MINIMIZE
)

# =========================
# SOLVE
# =========================

model.optimize()

# =========================
# RESULTATS
# =========================

theta_sol = np.array([theta[k].X for k in nodes])
U_sol     = np.array([U[k].X for k in nodes])

print("Status:", model.status)
print("Theta (first 10):", theta_sol[:10])
print("U (first 10):", U_sol[:10])

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information